# Simulador de Lançamento de Projétil (versão Colab)

Simulador interativo do lançamento de projéteis sem resistência do ar. Esta é a versão para Google Colab, feita com `ipywidgets`. A física é a mesma do `fisica.py` do repositório.

Como usar: execute as duas células de código abaixo, na ordem. A primeira define as funções de física e a segunda monta a interface. Depois é só mexer nos sliders e clicar em "Lançar".

Observação: ao clicar em "Lançar", a animação aparece no lugar do gráfico. Ao mexer em qualquer slider, o gráfico estático volta com os novos valores.


In [ ]:
# PARTE 1: FÍSICA (fórmulas analíticas, sem resistência do ar)
import numpy as np
import matplotlib.pyplot as plt


def calcular_resultados(v0, theta_graus, y0, g):
    """Retorna tempo de voo, altura máxima e alcance (fórmulas 3, 4 e 5 do enunciado)."""
    theta = np.radians(theta_graus)
    v0x = v0 * np.cos(theta)
    v0y = v0 * np.sin(theta)

    t_voo = (v0y + np.sqrt(v0y**2 + 2 * g * y0)) / g
    y_max = y0 + (v0y**2) / (2 * g)
    alcance = v0x * t_voo

    return {"t_voo": t_voo, "y_max": y_max, "alcance": alcance}


def calcular_trajetoria(v0, theta_graus, y0, g, x0=0, n_pontos=200):
    """Retorna os arrays (x, y) da trajetória, do lançamento até tocar o chão."""
    theta = np.radians(theta_graus)
    v0x = v0 * np.cos(theta)
    v0y = v0 * np.sin(theta)

    t_voo = calcular_resultados(v0, theta_graus, y0, g)["t_voo"]
    t = np.linspace(0, t_voo, n_pontos)

    x = x0 + v0x * t                      # fórmula 1
    y = y0 + v0y * t - 0.5 * g * t**2     # fórmula 2
    return x, y


In [ ]:
# PARTE 2: INTERFACE (sliders, botão e gráfico com ipywidgets)
#
# Fluxo de atualização:
#   mexer em um slider -> observe() chama atualizar()
#   atualizar() lê os sliders, chama as funções de física e redesenha o gráfico
#   clicar em "Lançar" -> on_click() chama lancar(), que mostra a animação
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from matplotlib.animation import FuncAnimation

# ----------------- Controles -----------------
estilo = {"description_width": "80px"}
layout = widgets.Layout(width="560px")


def criar_slider(valor, minimo, maximo, passo, descricao):
    return widgets.FloatSlider(
        value=valor, min=minimo, max=maximo, step=passo,
        description=descricao, style=estilo, layout=layout,
        continuous_update=True, readout_format=".1f",
    )


slider_v0 = criar_slider(40, 5, 150, 0.5, "v0 (m/s)")
slider_theta = criar_slider(45, 1, 89, 0.5, "θ (graus)")
slider_y0 = criar_slider(0, 0, 50, 0.5, "y0 (m)")
slider_g = criar_slider(9.8, 1.6, 24.8, 0.1, "g (m/s²)")
botao_lancar = widgets.Button(description="Lançar", button_style="primary", icon="play")

texto_resultados = widgets.HTML()   # mostra alcance, altura máxima e tempo de voo
area_grafico = widgets.Output()     # mostra o gráfico (ou a animação)


# ----------------- Funções auxiliares -----------------
def ler_parametros():
    return slider_v0.value, slider_theta.value, slider_y0.value, slider_g.value


def parametros_validos(v0, theta, y0, g):
    """Tratamento de entradas inválidas: evita contas sem sentido físico."""
    return v0 > 0 and 0 < theta < 90 and y0 >= 0 and g > 0


def desenhar_trajetoria(ax, x, y, resultados):
    """Desenha a curva, os pontos de destaque e formata o gráfico."""
    ax.plot(x, y, color="royalblue", linewidth=2)
    ax.scatter([x[np.argmax(y)]], [resultados["y_max"]],
               color="red", zorder=5, label="Altura máxima")
    ax.scatter([x[-1]], [y[-1]], color="green", zorder=5, label="Ponto de queda")

    # Limites com a mesma escala nos dois eixos (1 m em x = 1 m em y).
    # Os limites são definidos para a curva caber inteira e o formato da
    # caixa é ajustado pelo aspect "equal", então nada é cortado.
    xmin, xmax = min(0, x.min()), x.max()
    ymin, ymax = min(0, y.min()), max(y.max(), resultados["y_max"])
    margem = 0.06 * max(xmax - xmin, ymax - ymin) + 1
    ax.set_xlim(xmin - margem, xmax + margem)
    ax.set_ylim(ymin - margem, ymax + margem)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("Distância horizontal x (m)")
    ax.set_ylabel("Altura y (m)")
    ax.set_title("Trajetória do Projétil (sem resistência do ar)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")


# ----------------- Atualização em tempo real -----------------
def atualizar(_=None):
    v0, theta, y0, g = ler_parametros()

    with area_grafico:
        clear_output(wait=True)

        if not parametros_validos(v0, theta, y0, g):
            texto_resultados.value = (
                "<span style='color:red'>Entrada inválida: v0 e g devem ser maiores que 0 "
                "e o ângulo deve estar entre 0° e 90°.</span>"
            )
            return

        resultados = calcular_resultados(v0, theta, y0, g)
        x, y = calcular_trajetoria(v0, theta, y0, g)

        texto_resultados.value = (
            f"<b>Tempo de voo:</b> {resultados['t_voo']:.2f} s &nbsp;&nbsp; "
            f"<b>Altura máxima:</b> {resultados['y_max']:.2f} m &nbsp;&nbsp; "
            f"<b>Alcance:</b> {resultados['alcance']:.2f} m"
        )

        fig, ax = plt.subplots(figsize=(9, 4.5))
        desenhar_trajetoria(ax, x, y, resultados)
        plt.show()
        plt.close(fig)


# ----------------- Animação do botão "Lançar" -----------------
def lancar(_=None):
    v0, theta, y0, g = ler_parametros()
    if not parametros_validos(v0, theta, y0, g):
        return

    resultados = calcular_resultados(v0, theta, y0, g)
    x, y = calcular_trajetoria(v0, theta, y0, g)                    # curva completa
    x_anim, y_anim = calcular_trajetoria(v0, theta, y0, g, n_pontos=50)  # quadros

    fig, ax = plt.subplots(figsize=(9, 4.5))
    desenhar_trajetoria(ax, x, y, resultados)
    ponto, = ax.plot([], [], "o", color="black", markersize=9, zorder=6)

    def quadro(i):
        ponto.set_data([x_anim[i]], [y_anim[i]])
        return (ponto,)

    anim = FuncAnimation(fig, quadro, frames=len(x_anim), interval=40, blit=True)
    html = anim.to_jshtml(default_mode="once")   # toca uma vez, com controles
    plt.close(fig)

    with area_grafico:
        clear_output(wait=True)
        display(HTML(html))


# ----------------- Ligação dos controles às funções -----------------
for slider in (slider_v0, slider_theta, slider_y0, slider_g):
    slider.observe(atualizar, names="value")
botao_lancar.on_click(lancar)

display(widgets.VBox([slider_v0, slider_theta, slider_y0, slider_g,
                      botao_lancar, texto_resultados, area_grafico]))
atualizar()   # desenha o gráfico inicial
